In [ ]:
import matplotlib.pyplot as plt
from transformers import AutoModelForZeroShotObjectDetection
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

labels = ["crosswalk", "ebike_no", "ebike_ok", "human", "kickboard_no", "kickboard_ok", "parking_zone", "tactile_paving"]
processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(device)

base_data_dir = "../../data/labeled/parking/coco"

train_dataset = CocoGDINODataset(
    img_dir=f"{base_data_dir}/train",
    ann_file=f"{base_data_dir}/train/_annotations.coco.json",
    processor=processor,
    labels=labels
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()
epochs = 10

loss_history = []

for epoch in range(epochs):
    epoch_loss = 0
    total_batches = len(train_loader)
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        targets = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]
        
        outputs = model(**inputs, labels=targets)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == total_batches:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{total_batches}] | Loss: {loss.item():.4f}", flush=True)
        
    avg_epoch_loss = epoch_loss / total_batches
    loss_history.append(avg_epoch_loss)
    
    clear_output(wait=True)
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', color='blue', linestyle='-', linewidth=2)
    plt.title('Grounding DINO Fine-Tuning Train Loss (Real-time)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.xlim(1, epochs)
    plt.xticks(range(1, epochs + 1))
    plt.show()
    
    print(f"👉 Epoch {epoch+1} 완료! 평균 Loss: {avg_epoch_loss:.4f}", flush=True)
    print("-" * 50, flush=True)

model.save_pretrained("fine_tuned_gdino")
processor.save_pretrained("fine_tuned_gdino")
print("✅ 모델 저장 완료 ('fine_tuned_gdino')")

In [ ]:
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

fine_tuned_model = AutoModelForZeroShotObjectDetection.from_pretrained("fine_tuned_gdino").to(device)
fine_tuned_processor = AutoProcessor.from_pretrained("fine_tuned_gdino")

test_dir_path = f"{base_data_dir}/test"
text_prompt = ". ".join(labels) + "."

fine_tuned_model.eval()
for file_name in os.listdir(test_dir_path):
    if not file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
        
    image_path = os.path.join(test_dir_path, file_name)
    image = Image.open(image_path).convert("RGB")
    
    inputs = fine_tuned_processor(images=image, text=text_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = fine_tuned_model(**inputs)
        
    results = fine_tuned_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.3,
        text_threshold=0.25,
        target_sizes=[image.size[::-1]]
    )[0]
    
    draw = ImageDraw.Draw(image)
    for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
        box = [int(i) for i in box.tolist()]
        draw.rectangle(box, outline="blue", width=3)
        draw.text((box[0], box[1] - 10), f"{label}: {score:.2f}", fill="blue")
        
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis("off")
    plt.show()